# Calculate Design Temperature for Postal Codes

This notebook calculates design temperatures for postal code areas.

**Process:**
1. Consolidate all design temperature files from stations into a single CSV
2. Read postal code shapefile (plz-5stellig.shp)
3. Calculate centroid (geometric center) for each postal code polygon
4. Extract postal code and location information
5. Use IDW-KNN interpolation to find design temperatures for postal code centroids (to be implemented)


In [ ]:
!uv pip install geopy


Using Python 3.11.14 environment at: /home/abhishek/Documents/heatpump_ai_2/climate_data_prep/.venv
Resolved 2 packages in 297ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/122.50 KiB          
⠙ Preparing packages... (0/2)-------------------     0 B/122.50 KiB          
geographiclib        ------------------------------     0 B/39.79 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/122.50 KiB          
geographiclib        ------------------------------     0 B/39.79 KiB
⠙ Preparing packages... (0/2)------------------- 16.00 KiB/122.50 KiB        
geographiclib        ------------------------------ 6.78 KiB/39.79 KiB
⠙ Preparing packages... (0/2)------------------- 16.00 KiB/122.50 KiB        
geographiclib        ------------------------------ 6.78 KiB/39.79 KiB
⠙ Preparing packages... (0/2)------------------- 32.00 KiB/122.50 KiB        

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from geopy.distance import geodesic


In [16]:
# Configuration
DESIGN_TEMP_BASE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_design_temperature")
DESIGN_TEMP_CONSOLIDATED_FILE = Path("/mnt/d/heatpump_data/climate_data/dwd_design_temperature_consolidated.csv")

SHAPEFILE_PATH = Path("/mnt/d/heatpump_data/postal_code_data/postal_code_map/plz-5stellig.shp")
POSTAL_CODE_CENTROIDS_FILE = Path("/mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv")
POSTAL_CODE_DESIGN_TEMP_FILE = Path("/mnt/d/heatpump_data/postal_code_data/postal_code_design_temperature.csv")

# IDW-KNN Interpolation Parameters
INITIAL_DISTANCE_KM = 25  # Initial search radius (km) - "safe zone" for most of Germany
MAX_DISTANCE_KM = 50  # Maximum distance to consider stations (km) - expands if needed
MIN_K = 3  # Minimum number of neighbors required
MAX_K = 8  # Maximum number of neighbors to use
IDW_POWER = 2  # Power parameter for IDW (higher = more weight to closer stations)

print(f"Design temperature base directory: {DESIGN_TEMP_BASE_DIR}")
print(f"Design temperature base directory exists: {DESIGN_TEMP_BASE_DIR.exists()}")
print(f"Consolidated design temperature file: {DESIGN_TEMP_CONSOLIDATED_FILE}")
print(f"\nShapefile path: {SHAPEFILE_PATH}")
print(f"Shapefile exists: {SHAPEFILE_PATH.exists()}")
print(f"Postal code centroids file: {POSTAL_CODE_CENTROIDS_FILE}")
print(f"\nInterpolation parameters:")
print(f"  Initial distance: {INITIAL_DISTANCE_KM} km (expands to {MAX_DISTANCE_KM} km if needed)")
print(f"  Min neighbors: {MIN_K}")
print(f"  Max neighbors: {MAX_K}")
print(f"  IDW power: {IDW_POWER}")


Design temperature base directory: /mnt/d/heatpump_data/climate_data/dwd_design_temperature
Design temperature base directory exists: True
Consolidated design temperature file: /mnt/d/heatpump_data/climate_data/dwd_design_temperature_consolidated.csv

Shapefile path: /mnt/d/heatpump_data/postal_code_data/postal_code_map/plz-5stellig.shp
Shapefile exists: True
Postal code centroids file: /mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv

Interpolation parameters:
  Initial distance: 25 km (expands to 50 km if needed)
  Min neighbors: 3
  Max neighbors: 8
  IDW power: 2


## Step 1: Consolidate Design Temperature Files

Collect all design temperature files from station subdirectories into a single CSV file.


In [3]:
# Find all design temperature CSV files
print("Finding all design temperature files...")
design_temp_files = list(DESIGN_TEMP_BASE_DIR.rglob("*.csv"))

print(f"Found {len(design_temp_files)} design temperature files")

if len(design_temp_files) == 0:
    print("Warning: No design temperature files found!")
else:
    print(f"\nSample files:")
    for f in design_temp_files[:5]:
        print(f"  {f}")


Finding all design temperature files...
Found 746 design temperature files

Sample files:
  /mnt/d/heatpump_data/climate_data/dwd_design_temperature/stundenwerte_TU_00003_19500401_20110331_hist/produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
  /mnt/d/heatpump_data/climate_data/dwd_design_temperature/stundenwerte_TU_00003_19500401_20110331_hist/produkt_tu_stunde_19500401_20110331_3_2_lat_50_7827_lon_6_0941.csv
  /mnt/d/heatpump_data/climate_data/dwd_design_temperature/stundenwerte_TU_00044_20070401_20241231_hist/produkt_tu_stunde_20070401_20241231_44_3_lat_52_9164_lon_8_2211.csv
  /mnt/d/heatpump_data/climate_data/dwd_design_temperature/stundenwerte_TU_00071_20091201_20191231_hist/produkt_tu_stunde_20091201_20191231_71_2_lat_48_2156_lon_8_9784.csv
  /mnt/d/heatpump_data/climate_data/dwd_design_temperature/stundenwerte_TU_00073_20070401_20241231_hist/produkt_tu_stunde_20070401_20230920_73_1_lat_48_6159_lon_13_0506.csv


In [ ]:
# Consolidate all design temperature files into a single DataFrame
if len(design_temp_files) > 0:
    print("Reading and consolidating design temperature files...")
    all_design_temps = []
    
    for file_path in tqdm(design_temp_files, desc="Processing files"):
        try:
            df = pd.read_csv(file_path)
            if len(df) > 0:
                all_design_temps.append(df)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    if len(all_design_temps) > 0:
        consolidated_df = pd.concat(all_design_temps, ignore_index=True)
        print(f"\nConsolidated DataFrame shape: {consolidated_df.shape}")
        print(f"Columns: {consolidated_df.columns.tolist()}")
        print(f"\nFirst few rows:")
        print(consolidated_df.head())
        print(f"\nUnique stations: {consolidated_df['STATIONS_ID'].nunique()}")
    else:
        print("No valid design temperature data found!")
        consolidated_df = None
else:
    consolidated_df = None


Reading and consolidating design temperature files...


Processing files: 100%|██████████| 746/746 [00:03<00:00, 213.59it/s]



Consolidated DataFrame shape: (746, 9)
Columns: ['STATIONS_ID', 'latitude', 'longitude', 'design_temperature', 'lowest_10_rolling_avgs', 'date_range_start', 'date_range_end', 'total_days', 'days_with_rolling_avg']

First few rows:
  STATIONS_ID  latitude  longitude  design_temperature  \
0           3   50.7827     6.0941           -8.581250   
1         3_2   50.7827     6.0941           -8.581250   
2        44_3   52.9164     8.2211           -9.577083   
3        71_2   48.2156     8.9784          -14.887500   
4        73_1   48.6159    13.0506          -13.262500   

                              lowest_10_rolling_avgs  date_range_start  \
0  -8.58, -8.26, -7.92, -7.41, -7.36, -7.03, -6.3...          20030101   
1  -8.58, -8.26, -7.92, -7.41, -7.36, -7.03, -6.3...          20030101   
2  -9.58, -9.37, -9.25, -8.97, -8.53, -8.48, -8.3...          20070401   
3  -14.89, -14.69, -14.34, -14.26, -12.97, -12.93...          20091201   
4  -13.26, -12.99, -12.58, -12.52, -12.36, -12.30

In [5]:
# Save consolidated design temperature file
if consolidated_df is not None:
    print(f"\nSaving consolidated design temperatures to: {DESIGN_TEMP_CONSOLIDATED_FILE}")
    DESIGN_TEMP_CONSOLIDATED_FILE.parent.mkdir(parents=True, exist_ok=True)
    consolidated_df.to_csv(DESIGN_TEMP_CONSOLIDATED_FILE, index=False)
    
    print(f"✓ Saved {len(consolidated_df)} design temperature records")
    print(f"\nSummary statistics:")
    print(f"Design temperature range: {consolidated_df['design_temperature'].min():.2f} to {consolidated_df['design_temperature'].max():.2f} °C")
    print(f"Latitude range: {consolidated_df['latitude'].min():.6f} to {consolidated_df['latitude'].max():.6f}")
    print(f"Longitude range: {consolidated_df['longitude'].min():.6f} to {consolidated_df['longitude'].max():.6f}")
else:
    print("No consolidated data to save!")



Saving consolidated design temperatures to: /mnt/d/heatpump_data/climate_data/dwd_design_temperature_consolidated.csv
✓ Saved 746 design temperature records

Summary statistics:
Design temperature range: -27.13 to 0.82 °C
Latitude range: 47.398100 to 55.011000
Longitude range: 6.039200 to 14.951200


# Get centroids of postal codes

In [ ]:
# Read shapefile
print("Reading shapefile...")
gdf = gpd.read_file(SHAPEFILE_PATH)

print(f"Total postal codes: {len(gdf)}")
print(f"Columns: {gdf.columns.tolist()}")
print(f"\nFirst few rows:")
print(gdf.head())
print(f"\nCRS: {gdf.crs}")


Reading shapefile...
Total postal codes: 8170
Columns: ['plz', 'note', 'einwohner', 'qkm', 'geometry']

First few rows:
     plz                            note  einwohner        qkm  \
0  81248                  81248 MÃ¼nchen        121   1.984763   
1  60315  60315 Frankfurt am Main (FOUR)          0   0.017481   
2  24988                  24988 Oeversee       3350  36.491463   
3  93185         93185 Michelsneukirchen       1786  32.873844   
4  93489                93489 Schorndorf       2622  38.597260   

                                            geometry  
0  POLYGON ((11.39468 48.14729, 11.3949 48.1478, ...  
1  POLYGON ((8.67254 50.11264, 8.67259 50.11264, ...  
2  POLYGON ((9.36586 54.69994, 9.36683 54.70014, ...  
3  POLYGON ((12.47666 49.13598, 12.47702 49.13637...  
4  POLYGON ((12.54904 49.19318, 12.54953 49.19371...  

CRS: EPSG:4326


In [ ]:
# Calculate centroids
print("Calculating centroids...")
gdf['centroid'] = gdf.geometry.centroid
gdf['centroid_lon'] = gdf['centroid'].x
gdf['centroid_lat'] = gdf['centroid'].y

print(f"Centroids calculated for {len(gdf)} postal codes")
print(f"\nSample centroids:")
print(gdf[['plz', 'centroid_lon', 'centroid_lat']].head() if 'plz' in gdf.columns else gdf[['centroid_lon', 'centroid_lat']].head())


Calculating centroids...
Centroids calculated for 8170 postal codes

Sample centroids:
     plz  centroid_lon  centroid_lat
0  81248     11.403147     48.148273
1  60315      8.673922     50.112310
2  24988      9.429714     54.707718
3  93185     12.541324     49.121808
4  93489     12.596062     49.167521


/tmp/ipykernel_15974/2800204479.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centroid'] = gdf.geometry.centroid


In [ ]:
# Prepare output DataFrame
# Extract postal code column (may be named differently)
postal_code_col = None
for col in ['plz', 'PLZ', 'postal_code', 'POSTAL_CODE', 'postcode']:
    if col in gdf.columns:
        postal_code_col = col
        break

if postal_code_col is None:
    print("Warning: Could not find postal code column. Available columns:", gdf.columns.tolist())
    # Use index as postal code identifier
    centroids_df = pd.DataFrame({
        'postal_code': gdf.index.astype(str),
        'longitude': gdf['centroid_lon'],
        'latitude': gdf['centroid_lat']
    })
else:
    centroids_df = pd.DataFrame({
        'postal_code': gdf[postal_code_col].astype(str),
        'longitude': gdf['centroid_lon'],
        'latitude': gdf['centroid_lat']
    })

# Add any other relevant columns
if 'note' in gdf.columns:
    centroids_df['note'] = gdf['note']
elif 'ort' in gdf.columns:
    centroids_df['city'] = gdf['ort']

print(f"\nOutput DataFrame shape: {centroids_df.shape}")
print(f"\nFirst few rows:")
print(centroids_df.head())
print(f"\nData types:")
print(centroids_df.dtypes)



Output DataFrame shape: (8170, 4)

First few rows:
  postal_code  longitude   latitude                            note
0       81248  11.403147  48.148273                  81248 MÃ¼nchen
1       60315   8.673922  50.112310  60315 Frankfurt am Main (FOUR)
2       24988   9.429714  54.707718                  24988 Oeversee
3       93185  12.541324  49.121808         93185 Michelsneukirchen
4       93489  12.596062  49.167521                93489 Schorndorf

Data types:
postal_code     object
longitude      float64
latitude       float64
note            object
dtype: object


In [ ]:
# Save postal code centroids to CSV
print(f"\nSaving centroids to: {POSTAL_CODE_CENTROIDS_FILE}")
POSTAL_CODE_CENTROIDS_FILE.parent.mkdir(parents=True, exist_ok=True)
centroids_df.to_csv(POSTAL_CODE_CENTROIDS_FILE, index=False)

print(f"✓ Saved {len(centroids_df)} postal code centroids")
print(f"\nSummary statistics:")
print(f"Longitude range: {centroids_df['longitude'].min():.6f} to {centroids_df['longitude'].max():.6f}")
print(f"Latitude range: {centroids_df['latitude'].min():.6f} to {centroids_df['latitude'].max():.6f}")



Saving centroids to: /mnt/d/heatpump_data/postal_code_data/postal_code_centroids.csv
✓ Saved 8170 postal code centroids

Summary statistics:
Longitude range: 5.971716 to 14.982235
Latitude range: 47.368732 to 55.019700


## Step 3: Interpolate Design Temperatures for Postal Codes (IDW-KNN)

Use Inverse Distance Weighting with K-Nearest Neighbors to interpolate design temperatures from station locations to postal code centroids.

**Method:**
- **Dynamic radius approach**: Starts with 25 km radius (safe zone for most of Germany)
  - Expands to 50 km if fewer than MIN_K (3) stations found within 25 km
  - Ensures at least 3 stations are used when available
- Uses MIN_K to MAX_K nearest neighbors (3-8 stations)
- IDW weighting: w_i = 1 / (distance^power), where power=2
- Handles sparse areas with warnings

**Inputs:**
- Consolidated design temperature file with station locations
- Postal code centroids with coordinates

**Output:**
- Design temperature for each postal code
- Metadata: number of neighbors, distances, radius used, warnings for sparse areas


In [ ]:
# Load consolidated design temperature data
print("Loading consolidated design temperature data...")
if DESIGN_TEMP_CONSOLIDATED_FILE.exists():
    stations_df = pd.read_csv(DESIGN_TEMP_CONSOLIDATED_FILE)
    print(f"Loaded {len(stations_df)} stations")
    print(f"Columns: {stations_df.columns.tolist()}")
else:
    print(f"Error: Consolidated design temperature file not found at {DESIGN_TEMP_CONSOLIDATED_FILE}")
    stations_df = None

# Load postal code centroids
print("\nLoading postal code centroids...")
if POSTAL_CODE_CENTROIDS_FILE.exists():
    postal_codes_df = pd.read_csv(POSTAL_CODE_CENTROIDS_FILE)
    print(f"Loaded {len(postal_codes_df)} postal codes")
else:
    print(f"Error: Postal code centroids file not found at {POSTAL_CODE_CENTROIDS_FILE}")
    postal_codes_df = None


Loading consolidated design temperature data...
Loaded 746 stations
Columns: ['STATIONS_ID', 'latitude', 'longitude', 'design_temperature', 'lowest_10_rolling_avgs', 'date_range_start', 'date_range_end', 'total_days', 'days_with_rolling_avg']

Loading postal code centroids...
Loaded 8170 postal codes


In [ ]:
def calculate_distance_km(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two points using geodesic distance (Haversine formula).
    
    Args:
        lat1, lon1: Latitude and longitude of first point
        lat2, lon2: Latitude and longitude of second point
        
    Returns:
        Distance in kilometers
    """
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers


def idw_interpolate(postal_lat, postal_lon, stations_df, initial_distance_km, max_distance_km, min_k, max_k, power=2):
    """
    Interpolate design temperature using IDW-KNN with dynamic distance constraints.
    
    Uses a dynamic radius approach:
    - Starts with initial_distance_km (25 km) - "safe zone" for most areas
    - Expands to max_distance_km (50 km) if fewer than min_k stations found
    - Ensures at least min_k stations are used when available
    
    Args:
        postal_lat: Latitude of postal code centroid
        postal_lon: Longitude of postal code centroid
        stations_df: DataFrame with station data (columns: latitude, longitude, design_temperature)
        initial_distance_km: Initial search radius (km)
        max_distance_km: Maximum distance to consider stations (km)
        min_k: Minimum number of neighbors required
        max_k: Maximum number of neighbors to use
        power: Power parameter for IDW weighting
        
    Returns:
        Dictionary with interpolated design temperature and metadata including:
        - design_temperature: Interpolated design temperature
        - num_neighbors: Number of stations used
        - min_distance_km, max_distance_km: Distance range
        - radius_used_km: Radius used (25 or 50 km)
        - warning: Warning status if any
        - nearest_station_id: ID of nearest station
        - station_ids: Comma-separated list of all station IDs used
        - station_distances_km: Comma-separated list of distances (km)
        - station_design_temps: Comma-separated list of design temperatures
        - station_weights: Comma-separated list of IDW weights
    """
    # Calculate distances to all stations
    distances = []
    for _, station in stations_df.iterrows():
        dist_km = calculate_distance_km(
            postal_lat, postal_lon,
            station['latitude'], station['longitude']
        )
        distances.append({
            'distance_km': dist_km,
            'design_temperature': station['design_temperature'],
            'station_id': station['STATIONS_ID']
        })
    
    distances_df = pd.DataFrame(distances)
    
    # Try initial distance first (25 km)
    within_range = distances_df[distances_df['distance_km'] <= initial_distance_km].copy()
    radius_used = initial_distance_km
    
    # If we don't have enough stations, expand to max distance (50 km)
    if len(within_range) < min_k:
        within_range = distances_df[distances_df['distance_km'] <= max_distance_km].copy()
        radius_used = max_distance_km
    
    # If still no stations within max range, use nearest station anyway (with warning)
    if len(within_range) == 0:
        nearest = distances_df.nsmallest(1, 'distance_km')
        return {
            'design_temperature': nearest.iloc[0]['design_temperature'],
            'num_neighbors': 1,
            'min_distance_km': nearest.iloc[0]['distance_km'],
            'max_distance_km': nearest.iloc[0]['distance_km'],
            'radius_used_km': nearest.iloc[0]['distance_km'],
            'warning': 'no_stations_in_range',
            'nearest_station_id': nearest.iloc[0]['station_id'],
            'station_ids': str(nearest.iloc[0]['station_id']),
            'station_distances_km': f"{nearest.iloc[0]['distance_km']:.6f}",
            'station_design_temps': f"{nearest.iloc[0]['design_temperature']:.4f}",
            'station_weights': '1.0'
        }
    
    # Select k nearest neighbors (within distance threshold)
    # Use min_k if we have at least that many, otherwise use all available
    k = min(max_k, len(within_range))
    k = max(min_k, min(k, len(within_range)))  # Ensure we use at least min_k if available
    
    nearest_k = within_range.nsmallest(k, 'distance_km')
    
    # Handle case where we have fewer than min_k stations (even after expansion)
    if len(nearest_k) < min_k:
        # Use equal weights when we have fewer than min_k stations
        station_ids_str = ', '.join(nearest_k['station_id'].astype(str))
        station_distances_str = ', '.join(nearest_k['distance_km'].round(6).astype(str))
        station_temps_str = ', '.join(nearest_k['design_temperature'].round(4).astype(str))
        station_weights_str = ', '.join([f"{1.0/len(nearest_k):.6f}"] * len(nearest_k))
        
        return {
            'design_temperature': nearest_k['design_temperature'].mean(),
            'num_neighbors': len(nearest_k),
            'min_distance_km': nearest_k['distance_km'].min(),
            'max_distance_km': nearest_k['distance_km'].max(),
            'radius_used_km': radius_used,
            'warning': 'insufficient_neighbors',
            'nearest_station_id': nearest_k.iloc[0]['station_id'],
            'station_ids': station_ids_str,
            'station_distances_km': station_distances_str,
            'station_design_temps': station_temps_str,
            'station_weights': station_weights_str
        }
    
    # Calculate IDW weights: w_i = 1 / (d_i^power)
    # Avoid division by zero for exact matches
    nearest_k['weight'] = 1.0 / (nearest_k['distance_km'] ** power + 1e-10)
    
    # Calculate weighted average
    weighted_sum = (nearest_k['design_temperature'] * nearest_k['weight']).sum()
    weight_sum = nearest_k['weight'].sum()
    
    interpolated_temp = weighted_sum / weight_sum
    
    # Determine if we had to expand the radius
    warning = None
    if radius_used > initial_distance_km:
        warning = 'expanded_radius'
    
    # Format station details as comma-separated strings for CSV
    station_ids_str = ', '.join(nearest_k['station_id'].astype(str))
    station_distances_str = ', '.join(nearest_k['distance_km'].round(6).astype(str))
    station_temps_str = ', '.join(nearest_k['design_temperature'].round(4).astype(str))
    station_weights_str = ', '.join(nearest_k['weight'].round(6).astype(str))
    
    return {
        'design_temperature': interpolated_temp,
        'num_neighbors': len(nearest_k),
        'min_distance_km': nearest_k['distance_km'].min(),
        'max_distance_km': nearest_k['distance_km'].max(),
        'radius_used_km': radius_used,
        'warning': warning,
        'nearest_station_id': nearest_k.iloc[0]['station_id'],
        'station_ids': station_ids_str,
        'station_distances_km': station_distances_str,
        'station_design_temps': station_temps_str,
        'station_weights': station_weights_str
    }


## Step 3: Test Interpolation on a Single Postal Code

Test the IDW-KNN interpolation on a single postal code to verify the implementation before processing all postal codes.


In [25]:
# Test interpolation on a single postal code
if stations_df is not None and postal_codes_df is not None:
    print("=" * 80)
    print("TEST: Single Postal Code Interpolation")
    print("=" * 80)
    
    # Select a test postal code (you can change this to any postal code)
    # Using the first postal code as an example
    test_postal_code = postal_codes_df.iloc[0]
    
    print(f"\nTest Postal Code: {test_postal_code['postal_code']}")
    print(f"Location: {test_postal_code.get('note', 'N/A')}")
    print(f"Coordinates: ({test_postal_code['latitude']:.6f}, {test_postal_code['longitude']:.6f})")
    
    # Calculate distances to all stations
    print(f"\nCalculating distances to all {len(stations_df)} stations...")
    distances = []
    for _, station in stations_df.iterrows():
        dist_km = calculate_distance_km(
            test_postal_code['latitude'], test_postal_code['longitude'],
            station['latitude'], station['longitude']
        )
        distances.append({
            'station_id': station['STATIONS_ID'],
            'latitude': station['latitude'],
            'longitude': station['longitude'],
            'design_temperature': station['design_temperature'],
            'distance_km': dist_km
        })
    
    distances_df = pd.DataFrame(distances).sort_values('distance_km')
    
    # Show stations within different radii
    within_25km = distances_df[distances_df['distance_km'] <= INITIAL_DISTANCE_KM]
    within_50km = distances_df[distances_df['distance_km'] <= MAX_DISTANCE_KM]
    
    print(f"\nStations within {INITIAL_DISTANCE_KM} km: {len(within_25km)}")
    print(f"Stations within {MAX_DISTANCE_KM} km: {len(within_50km)}")
    
    if len(within_25km) > 0:
        print(f"\nNearest stations within {INITIAL_DISTANCE_KM} km:")
        print(within_25km[['station_id', 'distance_km', 'design_temperature']].head(10).to_string(index=False))
    
    if len(within_50km) > len(within_25km):
        print(f"\nAdditional stations between {INITIAL_DISTANCE_KM}-{MAX_DISTANCE_KM} km:")
        additional = within_50km[~within_50km['station_id'].isin(within_25km['station_id'])]
        print(additional[['station_id', 'distance_km', 'design_temperature']].head(10).to_string(index=False))
    
    # Run interpolation
    print(f"\n{'=' * 80}")
    print("Running IDW-KNN Interpolation...")
    print(f"{'=' * 80}")
    
    result = idw_interpolate(
        test_postal_code['latitude'],
        test_postal_code['longitude'],
        stations_df,
        INITIAL_DISTANCE_KM,
        MAX_DISTANCE_KM,
        MIN_K,
        MAX_K,
        IDW_POWER
    )
    
    # Show detailed results
    print(f"\n✓ Interpolation Results:")
    print(f"  Design Temperature: {result['design_temperature']:.2f} °C")
    print(f"  Number of neighbors used: {result['num_neighbors']}")
    print(f"  Radius used: {result['radius_used_km']} km")
    print(f"  Min distance: {result['min_distance_km']:.2f} km")
    print(f"  Max distance: {result['max_distance_km']:.2f} km")
    print(f"  Warning: {result['warning'] if result['warning'] else 'None'}")
    
    # Show which stations were actually used
    print(f"\nStations used in interpolation:")
    used_stations = distances_df.nsmallest(result['num_neighbors'], 'distance_km')
    used_stations = used_stations[used_stations['distance_km'] <= result['radius_used_km']]
    
    # Calculate weights for display
    used_stations['weight'] = 1.0 / (used_stations['distance_km'] ** IDW_POWER + 1e-10)
    used_stations['weight_pct'] = (used_stations['weight'] / used_stations['weight'].sum() * 100).round(2)
    used_stations['weighted_temp'] = used_stations['design_temperature'] * used_stations['weight']
    
    display_cols = ['station_id', 'distance_km', 'design_temperature', 'weight_pct']
    print(used_stations[display_cols].to_string(index=False))
    
    print(f"\nWeighted sum: {used_stations['weighted_temp'].sum():.4f}")
    print(f"Weight sum: {used_stations['weight'].sum():.4f}")
    print(f"Interpolated temperature: {used_stations['weighted_temp'].sum() / used_stations['weight'].sum():.2f} °C")
    
    print(f"\n{'=' * 80}")
    print("Test completed successfully!")
    print(f"{'=' * 80}")
    
else:
    print("Cannot run test - missing required data!")


TEST: Single Postal Code Interpolation

Test Postal Code: 81248
Location: 81248 MÃ¼nchen
Coordinates: (48.148273, 11.403147)

Calculating distances to all 746 stations...

Stations within 25 km: 3
Stations within 50 km: 22

Nearest stations within 25 km:
station_id  distance_km  design_temperature
      3139    10.982490          -11.687500
    3379_3    11.096282          -12.072917
      1515    11.664746          -10.015000

Additional stations between 25-50 km:
station_id  distance_km  design_temperature
     142_5    29.844091          -10.987500
     142_6    29.844091          -15.058333
     217_2    30.167543          -14.485417
    5538_3    34.589096          -14.145833
    5404_7    35.712451          -13.958333
    5404_8    35.712451          -13.718750
    2829_2    37.241508          -15.043750
    1262_3    37.996172          -14.164583
    1262_4    37.996172          -14.329167
    2829_1    38.551623          -13.485417

Running IDW-KNN Interpolation...

✓ Interpola

In [ ]:
# Interpolate design temperatures for all postal codes
if stations_df is not None and postal_codes_df is not None:
    print("\nInterpolating design temperatures for postal codes...")
    print(f"Processing {len(postal_codes_df)} postal codes...")
    print(f"Using dynamic radius: {INITIAL_DISTANCE_KM} km → {MAX_DISTANCE_KM} km if needed")
    
    results = []
    
    for idx, postal_code_row in tqdm(postal_codes_df.iterrows(), total=len(postal_codes_df), desc="Interpolating"):
        result = idw_interpolate(
            postal_code_row['latitude'],
            postal_code_row['longitude'],
            stations_df,
            INITIAL_DISTANCE_KM,
            MAX_DISTANCE_KM,
            MIN_K,
            MAX_K,
            IDW_POWER
        )
        
        results.append({
            'postal_code': postal_code_row['postal_code'],
            'latitude': postal_code_row['latitude'],
            'longitude': postal_code_row['longitude'],
            'design_temperature': result['design_temperature'],
            'num_neighbors': result['num_neighbors'],
            'min_distance_km': result['min_distance_km'],
            'max_distance_km': result['max_distance_km'],
            'radius_used_km': result['radius_used_km'],
            'warning': result['warning'],
            'nearest_station_id': result['nearest_station_id'],
            'station_ids': result['station_ids'],
            'station_distances_km': result['station_distances_km'],
            'station_design_temps': result['station_design_temps'],
            'station_weights': result['station_weights']
        })
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    print(f"\n✓ Interpolation complete!")
    print(f"\nResults summary:")
    print(f"Total postal codes processed: {len(results_df)}")
    print(f"Design temperature range: {results_df['design_temperature'].min():.2f} to {results_df['design_temperature'].max():.2f} °C")
    print(f"\nNeighbor statistics:")
    print(f"  Average neighbors used: {results_df['num_neighbors'].mean():.2f}")
    print(f"  Min neighbors: {results_df['num_neighbors'].min()}")
    print(f"  Max neighbors: {results_df['num_neighbors'].max()}")
    print(f"\nDistance statistics:")
    print(f"  Average min distance: {results_df['min_distance_km'].mean():.2f} km")
    print(f"  Average max distance: {results_df['max_distance_km'].mean():.2f} km")
    print(f"  Max distance to nearest station: {results_df['min_distance_km'].max():.2f} km")
    print(f"\nRadius usage:")
    radius_25 = (results_df['radius_used_km'] == INITIAL_DISTANCE_KM).sum()
    radius_50 = (results_df['radius_used_km'] == MAX_DISTANCE_KM).sum()
    print(f"  Postal codes using {INITIAL_DISTANCE_KM} km radius: {radius_25} ({radius_25/len(results_df)*100:.1f}%)")
    print(f"  Postal codes using {MAX_DISTANCE_KM} km radius: {radius_50} ({radius_50/len(results_df)*100:.1f}%)")
    
    # Check for warnings
    warnings_df = results_df[results_df['warning'].notna()]
    if len(warnings_df) > 0:
        print(f"\n⚠ Warnings:")
        print(f"  Postal codes with warnings: {len(warnings_df)}")
        print(f"  Breakdown by warning type:")
        print(warnings_df['warning'].value_counts())
        print(f"\n  Sample postal codes with warnings:")
        print(warnings_df[['postal_code', 'warning', 'radius_used_km', 'min_distance_km', 'num_neighbors']].head(10))
    else:
        print(f"\n✓ No warnings - all postal codes have sufficient nearby stations!")
    
else:
    print("Cannot proceed with interpolation - missing required data!")
    results_df = None



Interpolating design temperatures for postal codes...
Processing 8170 postal codes...
Using dynamic radius: 25 km → 50 km if needed


Interpolating: 100%|██████████| 8170/8170 [17:28<00:00,  7.79it/s]


✓ Interpolation complete!

Results summary:
Total postal codes processed: 8170
Design temperature range: -25.18 to -5.22 °C

Neighbor statistics:
  Average neighbors used: 5.61
  Min neighbors: 2
  Max neighbors: 8

Distance statistics:
  Average min distance: 11.20 km
  Average max distance: 25.42 km
  Max distance to nearest station: 43.84 km

Radius usage:
  Postal codes using 25 km radius: 6278 (76.8%)
  Postal codes using 50 km radius: 1892 (23.2%)

⚠ Warnings:
  Postal codes with warnings: 1892
  Breakdown by warning type:
warning
expanded_radius           1890
insufficient_neighbors       2
Name: count, dtype: int64

  Sample postal codes with warnings:
    postal_code          warning  radius_used_km  min_distance_km  \
3         93185  expanded_radius              50        32.677843   
4         93489  expanded_radius              50        33.756220   
5         93494  expanded_radius              50        25.062357   
6         93473  expanded_radius              50      

In [ ]:
# Save results
if results_df is not None:
    print(f"\nSaving results to: {POSTAL_CODE_DESIGN_TEMP_FILE}")
    POSTAL_CODE_DESIGN_TEMP_FILE.parent.mkdir(parents=True, exist_ok=True)
    
    # Save full results with metadata
    results_df.to_csv(POSTAL_CODE_DESIGN_TEMP_FILE, index=False)
    
    print(f"✓ Saved design temperatures for {len(results_df)} postal codes")
    
    # Also create a simplified version with just postal code and design temperature
    simplified_df = results_df[['postal_code', 'latitude', 'longitude', 'design_temperature']].copy()
    simplified_file = POSTAL_CODE_DESIGN_TEMP_FILE.parent / "postal_code_design_temperature_simple.csv"
    simplified_df.to_csv(simplified_file, index=False)
    print(f"✓ Saved simplified version to: {simplified_file}")
    
    # Save summary statistics to a text file
    summary_file = POSTAL_CODE_DESIGN_TEMP_FILE.parent / "interpolation_summary_statistics.txt"
    
    with open(summary_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("IDW-KNN Interpolation Summary Statistics\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("Interpolation Parameters:\n")
        f.write(f"  Initial distance: {INITIAL_DISTANCE_KM} km\n")
        f.write(f"  Maximum distance: {MAX_DISTANCE_KM} km\n")
        f.write(f"  Minimum neighbors: {MIN_K}\n")
        f.write(f"  Maximum neighbors: {MAX_K}\n")
        f.write(f"  IDW power: {IDW_POWER}\n\n")
        
        f.write("Results Summary:\n")
        f.write(f"Total postal codes processed: {len(results_df)}\n")
        f.write(f"Design temperature range: {results_df['design_temperature'].min():.2f} to {results_df['design_temperature'].max():.2f} °C\n\n")
        
        f.write("Neighbor Statistics:\n")
        f.write(f"  Average neighbors used: {results_df['num_neighbors'].mean():.2f}\n")
        f.write(f"  Min neighbors: {results_df['num_neighbors'].min()}\n")
        f.write(f"  Max neighbors: {results_df['num_neighbors'].max()}\n\n")
        
        f.write("Distance Statistics:\n")
        f.write(f"  Average min distance: {results_df['min_distance_km'].mean():.2f} km\n")
        f.write(f"  Average max distance: {results_df['max_distance_km'].mean():.2f} km\n")
        f.write(f"  Max distance to nearest station: {results_df['min_distance_km'].max():.2f} km\n\n")
        
        f.write("Radius Usage:\n")
        radius_25 = (results_df['radius_used_km'] == INITIAL_DISTANCE_KM).sum()
        radius_50 = (results_df['radius_used_km'] == MAX_DISTANCE_KM).sum()
        f.write(f"  Postal codes using {INITIAL_DISTANCE_KM} km radius: {radius_25} ({radius_25/len(results_df)*100:.1f}%)\n")
        f.write(f"  Postal codes using {MAX_DISTANCE_KM} km radius: {radius_50} ({radius_50/len(results_df)*100:.1f}%)\n\n")
        
        # Warning statistics
        warnings_df = results_df[results_df['warning'].notna()]
        if len(warnings_df) > 0:
            f.write("Warnings:\n")
            f.write(f"  Postal codes with warnings: {len(warnings_df)}\n")
            f.write(f"  Breakdown by warning type:\n")
            for warning_type, count in warnings_df['warning'].value_counts().items():
                f.write(f"    {warning_type}: {count}\n")
            f.write(f"\n  Sample postal codes with warnings:\n")
            sample_warnings = warnings_df[['postal_code', 'warning', 'radius_used_km', 'min_distance_km', 'num_neighbors']].head(10)
            f.write(sample_warnings.to_string(index=False))
        else:
            f.write("Warnings: None - all postal codes have sufficient nearby stations!\n")
    
    print(f"✓ Saved summary statistics to: {summary_file}")
    
    print(f"\nFirst few results:")
    print(results_df[['postal_code', 'design_temperature', 'num_neighbors', 'radius_used_km', 'min_distance_km', 'warning']].head(10))
else:
    print("No results to save!")



Saving results to: /mnt/d/heatpump_data/postal_code_data/postal_code_design_temperature.csv
✓ Saved design temperatures for 8170 postal codes
✓ Saved simplified version to: /mnt/d/heatpump_data/postal_code_data/postal_code_design_temperature_simple.csv

First few results:
   postal_code  design_temperature  num_neighbors  radius_used_km  \
0        81248          -11.301942              3              25   
1        60315          -10.400924              7              25   
2        24988           -9.055082              5              25   
3        93185          -12.970385              8              50   
4        93489          -13.697479              8              50   
5        93494          -14.928152              8              50   
6        93473          -15.842400              7              50   
7        99331          -17.250958              8              50   
8        60312          -10.398357              7              25   
9        98694          -17.272538  

## Step 5: Validate and Check Edge Cases

Load the simple CSV file and analyze edge cases - extreme values, outliers, and boundary conditions.


In [28]:
# Load and validate the simple CSV file
simple_file = POSTAL_CODE_DESIGN_TEMP_FILE.parent / "postal_code_design_temperature_simple.csv"

if simple_file.exists():
    print("Loading simple CSV file...")
    simple_df = pd.read_csv(simple_file)
    
    print(f"Loaded {len(simple_df)} postal codes")
    print(f"\nColumns: {simple_df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(simple_df.head())
    
    print(f"\n{'=' * 80}")
    print("BASIC STATISTICS")
    print(f"{'=' * 80}")
    print(f"Total postal codes: {len(simple_df)}")
    print(f"Design temperature range: {simple_df['design_temperature'].min():.2f} to {simple_df['design_temperature'].max():.2f} °C")
    print(f"Mean design temperature: {simple_df['design_temperature'].mean():.2f} °C")
    print(f"Median design temperature: {simple_df['design_temperature'].median():.2f} °C")
    print(f"Standard deviation: {simple_df['design_temperature'].std():.2f} °C")
    
    print(f"\n{'=' * 80}")
    print("EDGE CASES - EXTREME VALUES")
    print(f"{'=' * 80}")
    
    # Coldest (lowest) design temperatures
    print(f"\n🔵 COLDEST (Lowest) Design Temperatures:")
    coldest = simple_df.nsmallest(10, 'design_temperature')
    print(coldest[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    # Warmest (highest) design temperatures
    print(f"\n🔴 WARMEST (Highest) Design Temperatures:")
    warmest = simple_df.nlargest(10, 'design_temperature')
    print(warmest[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    print(f"\n{'=' * 80}")
    print("GEOGRAPHICAL EDGES")
    print(f"{'=' * 80}")
    
    # Northernmost postal codes
    print(f"\n📍 NORTHERNMOST Postal Codes:")
    northernmost = simple_df.nlargest(10, 'latitude')
    print(northernmost[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    # Southernmost postal codes
    print(f"\n📍 SOUTHERNMOST Postal Codes:")
    southernmost = simple_df.nsmallest(10, 'latitude')
    print(southernmost[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    # Westernmost postal codes
    print(f"\n📍 WESTERNMOST Postal Codes:")
    westernmost = simple_df.nsmallest(10, 'longitude')
    print(westernmost[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    # Easternmost postal codes
    print(f"\n📍 EASTERNMOST Postal Codes:")
    easternmost = simple_df.nlargest(10, 'longitude')
    print(easternmost[['postal_code', 'latitude', 'longitude', 'design_temperature']].to_string(index=False))
    
    print(f"\n{'=' * 80}")
    print("OUTLIER DETECTION")
    print(f"{'=' * 80}")
    
    # Calculate IQR for outlier detection
    Q1 = simple_df['design_temperature'].quantile(0.25)
    Q3 = simple_df['design_temperature'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"Q1 (25th percentile): {Q1:.2f} °C")
    print(f"Q3 (75th percentile): {Q3:.2f} °C")
    print(f"IQR: {IQR:.2f} °C")
    print(f"Lower bound (Q1 - 1.5*IQR): {lower_bound:.2f} °C")
    print(f"Upper bound (Q3 + 1.5*IQR): {upper_bound:.2f} °C")
    
    outliers_low = simple_df[simple_df['design_temperature'] < lower_bound]
    outliers_high = simple_df[simple_df['design_temperature'] > upper_bound]
    
    print(f"\nOutliers below lower bound: {len(outliers_low)}")
    if len(outliers_low) > 0:
        print("Low outliers:")
        print(outliers_low[['postal_code', 'latitude', 'longitude', 'design_temperature']].head(10).to_string(index=False))
    
    print(f"\nOutliers above upper bound: {len(outliers_high)}")
    if len(outliers_high) > 0:
        print("High outliers:")
        print(outliers_high[['postal_code', 'latitude', 'longitude', 'design_temperature']].head(10).to_string(index=False))
    
    print(f"\n{'=' * 80}")
    print("PERCENTILE DISTRIBUTION")
    print(f"{'=' * 80}")
    percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    print("Percentile | Design Temperature (°C)")
    print("-" * 40)
    for p in percentiles:
        value = simple_df['design_temperature'].quantile(p / 100)
        print(f"   {p:3d}%   | {value:8.2f}")
    
    print(f"\n{'=' * 80}")
    print("VALIDATION COMPLETE")
    print(f"{'=' * 80}")
    
else:
    print(f"Error: Simple CSV file not found at {simple_file}")
    print("Please run the interpolation and save steps first.")


Loading simple CSV file...
Loaded 8170 postal codes

Columns: ['postal_code', 'latitude', 'longitude', 'design_temperature']

First few rows:
   postal_code   latitude  longitude  design_temperature
0        81248  48.148273  11.403147          -11.301942
1        60315  50.112310   8.673922          -10.400924
2        24988  54.707718   9.429714           -9.055082
3        93185  49.121808  12.541324          -12.970385
4        93489  49.167521  12.596062          -13.697479

BASIC STATISTICS
Total postal codes: 8170
Design temperature range: -25.18 to -5.22 °C
Mean design temperature: -12.15 °C
Median design temperature: -12.27 °C
Standard deviation: 2.14 °C

EDGE CASES - EXTREME VALUES

🔵 COLDEST (Lowest) Design Temperatures:
 postal_code  latitude  longitude  design_temperature
       87645 47.567219  10.767517          -25.175183
       82475 47.409853  10.989176          -24.831213
       82491 47.454627  11.006812          -24.096138
       83730 47.736780  11.947953         